[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/mlp_olympic.ipynb)

# Olympic marathon: RBF vs Williams MLP (arcsin)

CIP-001D Phase F + CIP-0031. Same `pods` marathon data, two priors — squared-exponential versus the infinite-width MLP / **arcsin** kernel (Williams 1996). The RBF is stationary and lengthscale-driven; the MLP kernel comes from a Bayesian neural net with infinite hidden units and encodes angle geometry in input space (non-stationary in the usual SE sense).

**Scale note.** Years are \(O(2000)\). GPy’s MLP defaults are `weight_variance=bias_variance=variance=1` (we match that in code). With calendar years those defaults saturate the arcsin angles and the fit goes flat — same as GPy unless you standardise inputs or set `weight_variance ≈ 1/\mathrm{Var}(X)` yourself. RBF gets an explicit lengthscale in years. That year-scale MLP recipe is a **notebook** choice, not a library default.

Bands below use `predict` (\(y_*\)). Swap in `plot_1d(..., noiseless=True)` for latent \(f_*\). For a finite-\(M\) ridge view of the same MLP prior, see [weightspace_ridge_1d](./weightspace_ridge_1d.ipynb).

```bash
pip install pods matplotlib   # or: pip install 'gpc[examples]'
```


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc

try:
    from matplotlib import pyplot as plt
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from matplotlib import pyplot as plt
    from gpc.plot import plot_1d

try:
    import pods
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pods"])
    import pods


## Fit RBF and MLP on the same data

Initialise RBF with `lengthscale≈30` years. For MLP, keep GPy-like defaults then set `mlp.weight_variance = 1/Var(X)` so the arcsin kernel sees \(O(1)\) inputs. Compare objectives after SCG — different kernels, so do not expect identical numbers.


In [ ]:
import numpy as np
import gpc

try:
    import pods
except ImportError as exc:
    raise SystemExit("pip install pods  (or gpc[examples])") from exc

data = pods.datasets.olympic_marathon_men()
X = np.asarray(data["X"], dtype=float)
Y = np.asarray(data["Y"], dtype=float)
if X.ndim == 1:
    X = X[:, None]
if Y.ndim == 1:
    Y = Y[:, None]

m_rbf = gpc.GPRegression(X, Y, kernel="rbf")
m_rbf.set_by_path("rbf.lengthscale", 30.0)

m_mlp = gpc.GPRegression(X, Y, kernel="mlp")  # GPy-like defaults: wv=bv=var=1
# Year-scale recipe (not a library default — GPy does not auto-scale either):
m_mlp.set_by_path("mlp.weight_variance", 1.0 / float(np.var(X)))
m_mlp.set_by_path("mlp.bias_variance", 1.0)

print("MLP init weight_variance:", m_mlp.get_by_path("mlp.weight_variance"))

for m in (m_rbf, m_mlp):
    m.optimize(iters=150, messages=True)

print("RBF objective:", m_rbf.objective())
print("MLP objective:", m_mlp.objective())
print("RBF params:", {p: m_rbf.get_by_path(p) for p in m_rbf.param_paths()})
print("MLP params:", {p: m_mlp.get_by_path(p) for p in m_mlp.param_paths()})


## Side-by-side predictive bands

Stationary RBF smooths locally in year; the MLP / arcsin prior can look qualitatively different on the same marks, especially near the edges of the Olympic era. That geometry difference — not a claim that one “wins” — is the teaching point.


In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

Xtest = np.linspace(float(X.min()) - 4.0, float(X.max()) + 4.0, 400)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m_rbf, X, Y, Xtest=Xtest, ax=axes[0], label="RBF")
axes[0].set_title("RBF / SE")
axes[0].set_xlabel("year")
axes[0].set_ylabel("time (hours)")
plot_1d(m_mlp, X, Y, Xtest=Xtest, ax=axes[1], label="MLP (arcsin)")
axes[1].set_title("Williams MLP / arcsin")
axes[1].set_xlabel("year")
fig.suptitle("Olympic marathon — same data, different prior geometry")
plt.tight_layout()
plt.show()
